# Hands-on: Color Space, Thresholding, dan Background Replacement (Single Image Upload Edition)
### Study group IMV Laboratory.

Modul praktikum ini membahas konsep dan implementasi pemrosesan citra digital untuk segmentasi warna (*chroma keying*) menggunakan Python, OpenCV, dan Matplotlib.

> **Catatan Khusus Edisi Google Colab / Single Image Upload:**
> - Di Google Colab atau mesin cloud, kita tidak dapat mengakses webcam lokal secara langsung via hardware `cv2.VideoCapture(0)`.
> - Pada edisi ini, praktikan **hanya perlu mengunggah 1 berkas citra subjek** (misal foto diri atau objek di depan green screen / dinding warna polos).
> - **Tidak perlu mengunggah gambar background pengganti!** Citra background baru akan dibuat dan dibangkitkan secara otomatis melalui pemrograman (*code-generated background*) menggunakan array NumPy dan fungsi grafis OpenCV. Ukuran background akan selalu presisi mengikuti dimensi foto yang diunggah.

---

### Alur Pembelajaran:
1. **Pengenalan OpenCV & Akuisisi Citra (Upload 1 Gambar)**: Membaca berkas citra subjek menggunakan `cv2.imread` / widget upload Google Colab, membangkitkan citra background sintetis secara terprogram, serta mekanisme fallback otomatis.
2. **Eksplorasi Color Space**: Membedah perbedaan representasi BGR vs RGB vs HSV, serta memvisualisasikan channel Hue (H), Saturation (S), dan Value (V).
3. **Thresholding Dasar vs HSV Range Thresholding**: Membandingkan keterbatasan Binary Thresholding pada grayscale (`cv2.threshold`) dengan Color Range Thresholding pada HSV (`cv2.inRange`).
4. **Masking & Operasi Bitwise**: Memahami prinsip gerbang biner dan menerapkan `cv2.bitwise_not` serta `cv2.bitwise_and` untuk memisahkan subjek dari background lama.
5. **Background Replacement & Fungsi Pipeline Mandiri**: Menggabungkan subjek ke background baru menggunakan `cv2.add`, serta fungsi otomatisasi yang hanya membutuhkan 1 foto masukan.
6. **Latihan Praktikan**: Tantangan aplikatif mandiri untuk menguji pemahaman materi.

---

### Library yang Dipakai:
- **`numpy`**: Manipulasi array matriks citra, pembuatan batas warna (*lower* & *upper bound*), dan pembangkitan citra background terprogram.
- **`opencv-python` (`cv2`)**: Pembacaan berkas citra (`imread`), konversi ruang warna (`cvtColor`), thresholding (`threshold`, `inRange`), dan operasi bitwise (`bitwise_and`, `bitwise_not`, `add`).
- **`matplotlib`**: Menampilkan plot gambar dan perbandingan tahapan proses.

```bash
# Perintah instalasi library jika belum terpasang di environment Anda:
pip install numpy opencv-python matplotlib
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Deteksi apakah kode berjalan di Google Colab atau Local
try:
    from google.colab import files
    IN_COLAB = True
    print("[INFO] Berjalan di Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("[INFO] Berjalan di Local environment.")

# Konfigurasi resolusi visualisasi matplotlib agar tajam dan proporsional
plt.rcParams['figure.dpi'] = 100

## 0. Persiapan: Fungsi Bantu, Unggah 1 Citra Subjek, dan Pembuatan Background Baru

### Representasi Citra pada OpenCV:
- **Urutan Channel BGR:** Secara default, OpenCV membaca citra dalam urutan channel **BGR (Blue, Green, Red)**, bukan RGB. Citra disimpan di memori sebagai array `numpy.ndarray` bertipe `uint8` dengan rentang nilai $0 - 255$. Jika citra BGR langsung ditampilkan dengan pustaka Matplotlib (yang mengasumsikan urutan RGB), warna akan tampak terbalik (warna kulit menjadi kebiruan dan warna merah tertukar dengan warna biru).
- **Alur Kerja Bagian Ini (Single Upload):**
  1. Praktikan cukup mengunggah **1 foto subjek** berlatar belakang satu warna (misal kain hijau green screen, kain biru, atau dinding kamar polos) dengan nama berkas `image.jpg` atau `person.jpg`.
  2. Gambar background baru **tidak perlu dicari atau diunggah terpisah**. Kode di bawah langsung membangkitkan background pemandangan gradasi estetis secara terprogram dengan dimensi matriks yang otomatis presisi menyamai foto subjek ($H \times W$).
  3. **Mekanisme Fallback Otomatis:** Jika Anda belum sempat mengunggah foto, sistem secara otomatis membangkitkan citra subjek sintetis berlatar hijau berukuran $480 \times 640$, sehingga notebook dijamin **100% reproducible** dan langsung dapat dieksekusi tanpa error.

In [ ]:
def show(images, titles, cols=3, size=(4, 4)):
    """
    Fungsi bantu untuk menampilkan kumpulan gambar berdampingan menggunakan matplotlib.pyplot.
    Mendukung tampilan citra RGB (3 channel) maupun Grayscale/Mask (2 channel dengan cmap='gray', vmin=0, vmax=255).
    """
    n = len(images)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size[0], rows * size[1]))
    
    # Ratakan axes menjadi array 1D agar konsisten saat diiterasi
    if n == 1:
        axes = np.array([axes])
    else:
        axes = np.array(axes).flatten()
        
    for i in range(n):
        img = images[i]
        title = titles[i] if i < len(titles) else ""
        ax = axes[i]
        
        # Deteksi dimensi gambar: 2 channel (Grayscale/Mask) atau 3 channel (RGB)
        if img.ndim == 2:
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(img)
            
        ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
        ax.axis('off')
        
    # Sembunyikan subplot sisa yang tidak terpakai
    for j in range(n, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

# ==============================================================================
# 1. Pilihan Upload 1 Foto Subjek (Khusus Google Colab)
# ==============================================================================
# Ubah UPLOAD_NOW = True jika ingin memunculkan tombol upload berkas di Google Colab
UPLOAD_NOW = False

if IN_COLAB and UPLOAD_NOW:
    print("Silakan unggah 1 foto subjek berlatar warna (misal kain hijau/biru/dinding polos):")
    uploaded = files.upload()

# ==============================================================================
# 2. Pemanggilan Citra Subjek (dengan Fallback Citra Sintetis)
# ==============================================================================
# Periksa ketersediaan file foto (bisa bernama image.jpg atau person.jpg)
target_paths = ['image.jpg', 'person.jpg']
frame = None
loaded_path = None

for path in target_paths:
    if os.path.exists(path):
        frame = cv2.imread(path)
        if frame is not None:
            loaded_path = path
            break

# Fallback: Buat citra sintetis jika belum ada file foto yang diunggah
if frame is None:
    print("[INFO] Foto belum diunggah. Menggunakan citra sintetis subjek berlatar hijau...")
    frame = np.zeros((480, 640, 3), dtype=np.uint8)
    # Latar hijau green screen (BGR: B=0, G=200, R=0)
    frame[:] = [0, 200, 0]
    
    # Objek simulasi subjek di tengah frame
    # Badan / baju biru (BGR: B=180, G=50, R=50)
    cv2.ellipse(frame, (320, 440), (140, 160), 0, 180, 360, (180, 50, 50), -1)
    # Wajah / warna kulit (BGR: B=140, G=180, R=235)
    cv2.circle(frame, (320, 230), 80, (140, 180, 235), -1)
    # Rambut hitam
    cv2.ellipse(frame, (320, 180), (82, 50), 0, 180, 360, (20, 20, 20), -1)
    # Mata dan senyum
    cv2.circle(frame, (290, 220), 8, (255, 255, 255), -1)
    cv2.circle(frame, (290, 220), 4, (20, 20, 20), -1)
    cv2.circle(frame, (350, 220), 8, (255, 255, 255), -1)
    cv2.circle(frame, (350, 220), 4, (20, 20, 20), -1)
    cv2.ellipse(frame, (320, 260), (30, 15), 0, 0, 180, (40, 40, 200), -1)
else:
    print(f"[INFO] Berhasil memuat citra subjek dari '{loaded_path}'.")

# ==============================================================================
# 3. Pembuatan Background Pengganti Secara Terprogram (Tanpa Perlu Upload)
# ==============================================================================
# Ukuran background baru otomatis mengikuti dimensi foto subjek secara presisi
h, w = frame.shape[:2]
bg_img = np.zeros((h, w, 3), dtype=np.uint8)

# Gradasi senja estetis (sunset gradient)
for y in range(h):
    ratio = y / h
    # BGR format: Biru-toska di atas bertransisi ke oranye hangat di bawah
    bg_img[y, :] = [
        int(235 * (1 - ratio) + 50 * ratio),
        int(150 * (1 - ratio) + 110 * ratio),
        int(60 * (1 - ratio) + 240 * ratio)
    ]

# Elemen grafis: Matahari buatan dengan pendar cahaya
cv2.circle(bg_img, (int(w * 0.78), int(h * 0.22)), int(min(h, w) * 0.12), (180, 240, 255), -1)
# Siluet bukit pemandangan di bagian bawah
cv2.ellipse(bg_img, (int(w * 0.25), int(h * 1.1)), (int(w * 0.6), int(h * 0.35)), 0, 0, 360, (40, 50, 40), -1)
cv2.ellipse(bg_img, (int(w * 0.8), int(h * 1.15)), (int(w * 0.5), int(h * 0.35)), 0, 0, 360, (25, 35, 25), -1)

# Konversi BGR ke RGB untuk visualisasi matplotlib
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
bg_rgb = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)

# Cetak metadata matriks citra frame
print(f"frame.shape : {frame.shape} -> (Tinggi: {frame.shape[0]}, Lebar: {frame.shape[1]}, Channel: {frame.shape[2]})")
print(f"frame.dtype : {frame.dtype}")
print(f"frame.min() : {frame.min()}")
print(f"frame.max() : {frame.max()}")

# Tampilkan visualisasi perbandingan
show(
    [frame, frame_rgb, bg_rgb],
    [
        "Foto Subjek (BGR tanpa konversi)",
        "Foto Subjek (RGB benar)",
        "Background Baru (Dibuat Program)"
    ],
    cols=3,
    size=(4, 4)
)

## 1. Color Space: RGB vs HSV

### Kelemahan Ruang Warna RGB/BGR untuk Segmentasi Warna:
- Pada ruang warna RGB/BGR, setiap warna dibentuk oleh kombinasi tiga channel: **Red**, **Green**, dan **Blue**.
- **Masalah mendasar:** Informasi **warna murni (kromatisitas)** dan **intensitas cahaya (luminans/kecerahan)** saling bercampur di seluruh channel R, G, dan B.
- Jika intensitas pencahayaan di ruangan berubah sedikit saja (misalnya muncul bayangan tubuh pada layar hijau atau perubahan sudut lampu), nilai ketiga channel R, G, dan B akan berubah serentak secara signifikan. Hal ini membuat penetapan ambang batas segmentasi (*thresholding*) pada ruang RGB menjadi sangat tidak stabil dan rentan gagal.

---

### Ruang Warna HSV (Hue, Saturation, Value):
Ruang warna HSV memisahkan komponen warna dari tingkat pencahayaan, menyerupai cara manusia mendeskripsikan warna. Di OpenCV dengan tipe data `uint8`, ketiga channel memiliki rentang nilai sebagai berikut:

- **Hue (H) — Jenis Warna Dasar:**
  - Menyatakan corak warna murni pada roda spektrum warna lingkaran ($0^{\circ} - 360^{\circ}$).
  - Di OpenCV, karena nilai `uint8` dibatasi hingga 255, rentang derajat dibagi dua menjadi **`0 - 179`**.
  - *Perkiraan nilai Hue di OpenCV:* Merah (`0 - 10` dan `170 - 179`), Kuning (`20 - 35`), **Hijau (`35 - 85`)**, Biru (`90 - 130`).
- **Saturation (S) — Kepekatan / Kemurnian Warna:**
  - Menyatakan seberapa jenuh warna tersebut, dari warna pudar/abu-abu ke warna murni pekat.
  - Rentang di OpenCV: **`0 - 255`**.
- **Value (V) — Kecerahan Cahaya (Brightness):**
  - Menyatakan intensitas cahaya dari gelap gulita/hitam hingga terang benderang.
  - Rentang di OpenCV: **`0 - 255`**.

> **Keuntungan Utama:** Untuk mengisolasi latar belakang hijau, kita cukup mengunci nilai pada channel **Hue**, sementara channel **Saturation** dan **Value** dapat diberi rentang batas toleransi yang lebar sehingga segmentasi tetap konsisten meskipun terdapat variasi bayangan pencahayaan.

In [ ]:
# Konversi citra frame dari BGR ke HSV
frame_hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# Pisahkan masing-masing channel Hue, Saturation, dan Value
h_channel, s_channel, v_channel = cv2.split(frame_hsv)

# Visualisasikan citra asli (RGB) bersama masing-masing channel HSV
# Channel H, S, dan V adalah matriks 2D sehingga otomatis dirender dengan cmap='gray'
show(
    [frame_rgb, h_channel, s_channel, v_channel],
    [
        "Gambar Asli (RGB)",
        "Channel Hue (H)",
        "Channel Saturation (S)",
        "Channel Value (V)"
    ],
    cols=4,
    size=(4, 4)
)

## 2. Thresholding: Binary Threshold vs HSV Range Thresholding

### Konsep Dasar Thresholding:
Thresholding adalah teknik segmentasi paling mendasar yang mengubah citra bergradasi kontinu menjadi citra biner (hanya bernilai `0` untuk hitam dan `255` untuk putih). Citra biner inilah yang berfungsi sebagai **mask** (stensil) pemisah antara objek yang diinginkan dan latar belakang yang ingin diganti.

---

### Perbandingan Metode:

- **`cv2.threshold()` (Binary Thresholding pada Citra Grayscale):**
  - Hanya bekerja pada $1$ channel (citra grayscale).
  - Citra grayscale hanya merepresentasikan intensitas kecerahan total:
    $$Y = 0.299R + 0.587G + 0.114B$$
  - Dua warna yang berbeda sama sekali (misal warna kulit subjek dan warna kain hijau latar) bisa saja menghasilkan intensitas kecerahan abu-abu yang persis sama. Akibatnya, `cv2.threshold()` **gagal memisahkan warna spesifik latar belakang**.
- **`cv2.inRange(hsv, lower, upper)` (Color Range Thresholding pada Ruang Warna HSV):**
  - Bekerja pada $3$ channel (H, S, dan V) secara simultan.
  - Memeriksa apakah setiap piksel berada di dalam rentang `lower <= pixel <= upper`.
  - Jika berada dalam rentang pada seluruh channel, piksel dijadikan `255` (putih); jika tidak, dijadikan `0` (hitam).
  - **Sangat presisi dalam mengisolasi warna latar hijau green screen** tanpa terpengaruh oleh warna objek di sekitarnya.

In [ ]:
# 1. Binary Thresholding konvensional pada citra Grayscale
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
_, binary_thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# 2. Color Range Thresholding pada HSV menggunakan cv2.inRange
# Definisikan rentang warna hijau (Green Screen):
# - Hue: 35 s.d. 85 (mencakup spektrum hijau muda, hijau daun, hingga hijau toska)
# - Saturation: 50 s.d. 255 (menyaring warna putih/abu-abu netral)
# - Value: 50 s.d. 255 (menyaring area bayangan yang terlalu gelap)
lower_green = np.array([35, 50, 50])
upper_green = np.array([85, 255, 255])

# Panduan penyesuaian jika menggunakan latar warna lain:
# - Latar Biru  : lower = np.array([90, 50, 50]),   upper = np.array([130, 255, 255])
# - Latar Merah : lower = np.array([0, 70, 50]),    upper = np.array([10, 255, 255])

# Buat mask biner: area hijau menjadi 255 (putih), area selain hijau menjadi 0 (hitam)
mask = cv2.inRange(frame_hsv, lower_green, upper_green)

# Tampilkan perbandingan ketiga hasil thresholding
show(
    [gray, binary_thresh, mask],
    [
        "Citra Grayscale",
        "Binary Threshold Biasa",
        "HSV Mask (cv2.inRange)"
    ],
    cols=3,
    size=(4, 4)
)

## 3. Masking & Operasi Bitwise

### Cara Kerja Matriks Mask Biner:
Matriks mask biner berfungsi sebagai stensil atau gerbang logika pada setiap piksel:
- **Piksel bernilai `255` (Putih / TRUE):** Meloloskan nilai piksel citra asli apa adanya.
- **Piksel bernilai `0` (Hitam / FALSE):** Menahan/menghapus piksel citra sehingga bernilai `0` (hitam pekat).

---

### Tahapan Pemotongan dengan Operasi Bitwise:
1. **`mask` (Mask Background):**
   - Area latar belakang lama bernilai `255` (putih) dan area subjek bernilai `0` (hitam).
   - Digunakan untuk memotong background baru buatan program agar hanya mengisi area latar hijau.
2. **`mask_inv = cv2.bitwise_not(mask)` (Mask Inverse / Subjek):**
   - Membalik nilai biner ($255 - \text{mask}$) sehingga area subjek bernilai `255` (putih) dan area latar hijau bernilai `0` (hitam).
   - Digunakan untuk memotong subjek dari frame citra.
3. **`cv2.bitwise_and(src, src, mask=...)`:**
   - Mengekstrak piksel gambar sesuai area putih pada mask yang ditentukan.

In [ ]:
# 1. Hitung mask inverse untuk mengisolasi subjek
mask_inv = cv2.bitwise_not(mask)

# 2. Ekstrak foreground (subjek) dari foto masukan
# Piksel subjek diloloskan, area green screen diubah menjadi hitam (0)
fg = cv2.bitwise_and(frame_rgb, frame_rgb, mask=mask_inv)

# 3. Ekstrak background baru (buatan program) sesuai bentuk area green screen
# Piksel background baru diloloskan pada posisi hijau, area subjek diubah menjadi hitam (0)
bg_cut = cv2.bitwise_and(bg_rgb, bg_rgb, mask=mask)

# Visualisasikan keempat matriks hasil proses masking
show(
    [mask, mask_inv, fg, bg_cut],
    [
        "Mask Background",
        "Mask Inverse (Subjek)",
        "Hasil Potong Foreground (fg)",
        "Hasil Potong Background (bg_cut)"
    ],
    cols=4,
    size=(4, 4)
)

## 4. Penggabungan Akhir (Background Replacement) & Fungsi Pipeline Mandiri

### Penggabungan Matriks Citra:
Langkah terakhir dari pipeline Chroma Key adalah menjumlahkan matriks `fg` dan matriks `bg_cut` menggunakan operasi penambahan citra:
$$\text{Hasil Akhir} = \text{fg} + \text{bg\_cut}$$

- Pada matriks `fg`: area subjek berisi piksel subjek asli, sedangkan area latar bernilai `0` (hitam).
- Pada matriks `bg_cut`: area subjek bernilai `0` (hitam), sedangkan area latar berisi piksel gambar background baru buatan program.
- Penjumlahan piksel:
  - Pada area subjek: $\text{Piksel Subjek} + 0 = \text{Piksel Subjek}$
  - Pada area latar: $0 + \text{Piksel Background Baru} = \text{Piksel Background Baru}$
- Kita menggunakan fungsi **`cv2.add(fg, bg_cut)`** untuk menggabungkan kedua citra tersebut secara sempurna.

---

### Fungsi Pipeline Otomatis (Single Input Image):
Di bawah ini disertakan fungsi serbaguna `apply_chroma_key(image_path, ...)` yang membungkus seluruh tahapan di atas. Fungsi ini:
- **Hanya membutuhkan 1 foto masukan (`image_path`).**
- Otomatis membangkitkan citra background baru sesuai gaya yang dipilih (`'sunset'`, `'studio_blue'`, atau `'white'`) dengan dimensi yang langsung presisi menyamai foto masukan.
- Menampilkan visualisasi perbandingan dan menyimpan berkas akhir ke disk (`output_chroma_key.jpg`).

In [ ]:
# 1. Hitung hasil penggabungan akhir dengan cv2.add
final_result = cv2.add(fg, bg_cut)

# Tampilkan perbandingan akhir dengan show()
show(
    [frame_rgb, bg_rgb, final_result],
    [
        "Foto Asli Subjek",
        "Background Baru (Dibuat Program)",
        "Hasil Akhir Background Replacement"
    ],
    cols=3,
    size=(5, 4)
)

# ==============================================================================
# 2. Fungsi Mandiri Pipeline Chroma Key (Hanya Butuh 1 Foto Masukan)
# ==============================================================================
def apply_chroma_key(image_path='image.jpg', 
                     lower_hsv=np.array([35, 50, 50]), 
                     upper_hsv=np.array([85, 255, 255]),
                     bg_style='sunset',
                     output_path='output_chroma_key.jpg'):
    """
    Fungsi pipeline otomatis untuk mengganti latar belakang foto secara instan.
    Hanya memerlukan 1 berkas foto subjek! Background baru dibangkitkan otomatis oleh program.
    
    Parameter:
    - image_path  : path file citra subjek (1 foto)
    - lower_hsv   : batas bawah HSV [H, S, V]
    - upper_hsv   : batas atas HSV [H, S, V]
    - bg_style    : gaya background ('sunset', 'studio_blue', atau 'white')
    - output_path : nama file penyimpanan hasil akhir
    """
    # 1. Baca citra subjek
    src = cv2.imread(image_path)
    if src is None:
        print(f"[ERROR] Berkas foto '{image_path}' tidak ditemukan.")
        return None

    h, w = src.shape[:2]

    # 2. Bangkitkan background baru secara terprogram sesuai dimensi foto
    new_bg = np.zeros((h, w, 3), dtype=np.uint8)
    
    if bg_style == 'studio_blue':
        # Backdrop studio biru dengan efek radial vignette
        cy, cx = h // 2, w // 2
        y_grid, x_grid = np.ogrid[:h, :w]
        dist = np.sqrt((x_grid - cx)**2 + (y_grid - cy)**2)
        norm_dist = np.clip(dist / np.sqrt(cx**2 + cy**2), 0, 1)
        new_bg[:, :, 0] = (220 - 120 * norm_dist).astype(np.uint8)  # Blue
        new_bg[:, :, 1] = (140 - 100 * norm_dist).astype(np.uint8)  # Green
        new_bg[:, :, 2] = (50 - 40 * norm_dist).astype(np.uint8)    # Red
    elif bg_style == 'white':
        # Studio putih bersih
        new_bg[:] = [245, 245, 245]
    else:  # Default 'sunset'
        for y in range(h):
            r_ratio = y / h
            new_bg[y, :] = [
                int(235 * (1 - r_ratio) + 50 * r_ratio),
                int(150 * (1 - r_ratio) + 110 * r_ratio),
                int(60 * (1 - r_ratio) + 240 * r_ratio)
            ]
        cv2.circle(new_bg, (int(w * 0.78), int(h * 0.22)), int(min(h, w) * 0.12), (180, 240, 255), -1)

    # 3. Konversi ke HSV dan buat Mask
    hsv = cv2.cvtColor(src, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lower_hsv, upper_hsv)
    mask_inv = cv2.bitwise_not(mask)

    # 4. Potong Subjek dan Background baru
    sub_cut = cv2.bitwise_and(src, src, mask=mask_inv)
    bg_cut = cv2.bitwise_and(new_bg, new_bg, mask=mask)

    # 5. Gabungkan citra akhir
    result_bgr = cv2.add(sub_cut, bg_cut)

    # 6. Simpan hasil ke disk
    cv2.imwrite(output_path, result_bgr)
    print(f"[SUCCESS] Hasil berhasil disimpan ke '{output_path}'.")

    # 7. Tampilkan visualisasi perbandingan
    sub_rgb = cv2.cvtColor(src, cv2.COLOR_BGR2RGB)
    bg_rgb = cv2.cvtColor(new_bg, cv2.COLOR_BGR2RGB)
    res_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

    show(
        [sub_rgb, bg_rgb, res_rgb],
        ["Foto Subjek", f"Background ({bg_style})", "Hasil Background Replacement"],
        cols=3,
        size=(5, 4)
    )
    return result_bgr

# Contoh pemanggilan fungsi pipeline pada 1 foto:
# apply_chroma_key('image.jpg', bg_style='studio_blue')

## 5. Latihan Praktikan

Untuk memperdalam pemahaman dan keterampilan praktis, selesaikan 3 latihan aplikatif berikut (hanya membutuhkan 1 foto yang diunggah):

1. **Latihan 1: Unggah 1 Foto Diri & Kalibrasi Warna (Custom Color Range)**
   - Unggah 1 foto diri Anda di Google Colab di depan latar berwarna bebas (misalnya dinding kamar berwarna biru, kain merah, atau pakaian tertentu).
   - Cari rentang nilai `[H, S, V]` warna latar tersebut dengan memeriksa nilai piksel (`frame_hsv[y, x]`).
   - Ubah nilai `lower_green` dan `upper_green` menjadi batas warna baru Anda, panggil fungsi `apply_chroma_key('nama_foto_anda.jpg')`, dan buktikan bahwa background pengganti buatan program berhasil diterapkan secara rapi.

2. **Latihan 2: Reduksi Bintik Noise & Smoothing pada Foto Unggahan**
   - Pada kondisi foto nyata, pencahayaan yang tidak merata sering memicu kebocoran atau bintik-bintik noise (lubang kecil pada tubuh subjek atau bintik hitam pada background).
   - Eksplorasi penyesuaian batas `Saturation` dan `Value` pada `cv2.inRange`.
   - Terapkan teknik *smoothing* sederhana sebelum thresholding atau pada mask hasil thresholding, misalnya:
     ```python
     # Alternatif A: Blurring pada frame sebelum thresholding
     frame_blur = cv2.GaussianBlur(frame, (5, 5), 0)
     
     # Alternatif B: Median filter untuk menghilangkan salt-and-pepper noise pada mask
     clean_mask = cv2.medianBlur(mask, 5)
     ```
   - Amati bagaimana perbedaan kebersihan mask dan ketajaman tepi (*edge*) subjek setelah dihaluskan.

3. **Latihan 3: Logika Masking Terbalik (*Invisibility Cloak*)**
   - Balikkan logika masking (`mask` dan `mask_inv`): buatlah agar badan atau pakaian subjek yang berwarna tertentu justru digantikan oleh background sintetis buatan program, sementara latar belakang foto asli tetap dipertahankan.
   - Perhatikan efek visual tembus pandang (*invisibility*) yang dihasilkan, mirip dengan efek jubah tembus pandang pada film fiksi ilmiah!